In [184]:
import boto3
from dotenv import load_dotenv
import os
import sqlalchemy
import pymysql

load_dotenv()

aws_access_key_id = os.getenv('AWS_ACCESS_KEY_ID')
aws_secret_access_key = os.getenv('AWS_SECRET_ACCESS_KEY')
region_name = os.getenv('region_name')
master_user_password = os.getenv('master_db_password')
master_username = os.getenv('master_db_name')

# Créez une session boto3
session = boto3.Session(
    aws_access_key_id=aws_access_key_id,
    aws_secret_access_key=aws_secret_access_key,
    region_name=region_name
)

# Créez un client RDS
rds_client = session.client('rds')

# Paramètres de la base de données
db_instance_identifier = 'dbkayak'
db_instance_class = 'db.t4g.micro'  # Free Tier instance type
engine = 'mysql'
allocated_storage = 20  # Free Tier allows up to 20 GB

In [204]:
# Vérifiez si l'instance RDS existe déjà
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    print(f"L'instance RDS '{db_instance_identifier}' existe déjà.")
except rds_client.exceptions.DBInstanceNotFoundFault:
    # Créez la base de données RDS si elle n'existe pas
    try:
        response = rds_client.create_db_instance(
            DBInstanceIdentifier=db_instance_identifier,
            DBInstanceClass=db_instance_class,
            Engine=engine,
            MasterUsername=master_username,
            MasterUserPassword=master_user_password,
            AllocatedStorage=allocated_storage,
            BackupRetentionPeriod=7,  # Number of days to retain backups
            MultiAZ=False,  # Free Tier does not support Multi-AZ deployments
            PubliclyAccessible=True,  # Set to False if you don't want the DB to be publicly accessible
            StorageType='gp2',  # General Purpose SSD
        )
        print("Creating RDS instance...")
        print(response)
    except Exception as e:
        print(f"Error creating RDS instance: {e}")

L'instance RDS 'dbkayak' existe déjà.


In [185]:
# Obtenez les informations de l'instance RDS
try:
    db_instance_info = rds_client.describe_db_instances(DBInstanceIdentifier=db_instance_identifier)
    endpoint = db_instance_info['DBInstances'][0]['Endpoint']['Address']
    port = db_instance_info['DBInstances'][0]['Endpoint']['Port']
    name=db_instance_info['DBInstances'][0]['DBInstanceIdentifier']
    print(f"Name: {name}")  
    print(f"Endpoint: {endpoint}")
    print(f"Port: {port}")
except Exception as e:
    print(f"Error retrieving DB instance info: {e}")

Name: dbkayak
Endpoint: dbkayak.c3g8yqkisjyz.eu-west-3.rds.amazonaws.com
Port: 3306


In [193]:
from sqlalchemy import create_engine, text
from sqlalchemy.exc import OperationalError

new_database_name = 'dbkayak'

# Chaîne de connexion sans base de données pour la création de la base
base_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}"
database_connection_string = f"mysql+pymysql://{master_username}:{master_user_password}@{endpoint}:{port}/{new_database_name}"

# Créer le moteur SQLAlchemy sans base de données spécifique
base_engine = create_engine(base_connection_string)

def check_and_create_database(engine, db_name):
    """Vérifie si la base de données existe et la crée si elle n'existe pas."""
    database_exists_query = text(f"SELECT SCHEMA_NAME FROM INFORMATION_SCHEMA.SCHEMATA WHERE SCHEMA_NAME = :db_name")

    try:
        with engine.connect() as connection:
            # Vérifie si la base de données existe
            result = connection.execute(database_exists_query, {"db_name": db_name}).fetchone()
            if result:
                print(f"La base de données '{db_name}' existe déjà.")
            else:
                # Crée la base de données si elle n'existe pas
                connection.execute(text(f"CREATE DATABASE {db_name}"))
                print(f"Base de données '{db_name}' créée avec succès.")
    except OperationalError as e:
        print(f"Erreur lors de la vérification ou de la création de la base de données : {e}")

def connect_to_database(connection_url):
    """Essaie de se connecter à une base de données et gère les erreurs de connexion."""
    engine = create_engine(connection_url)
    try:
        with engine.connect() as connection:
            print("Connexion réussie à la base de données MySQL!")
    except OperationalError as e:
        print(f"Erreur lors de la connexion à la base de données : {e}")

# Vérifiez et créez la base de données si nécessaire
check_and_create_database(base_engine, new_database_name)

# Créer le moteur pour se connecter à la nouvelle base de données
connect_to_database(database_connection_string)


La base de données 'dbkayak' existe déjà.
Connexion réussie à la base de données MySQL!


In [188]:
import pandas as pd
import requests

In [200]:
url='https://tmopenlabbucket.s3.eu-west-3.amazonaws.com/City_Meteo_Rank_Booking.csv'

df = pd.read_csv(url,index_col=0)

In [201]:
df.head()

,City,City_latitude,City_longitude,City_CCM,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude
0,Aigues Mortes,43.566152,4.191540,0.896,"['Hôtel Le Médiéval', 'Maison Arthur', 'Au Cœu...",['https://www.booking.com/hotel/fr/le-medieval...,"[8.7, 9.0, 9.9, 9.5, 8.5, 9.2, 8.4, 9.1, 9.4, ...",['Le Médiéval is located on the banks of the c...,"[43.57186625, 43.5659428, 43.565401, 43.570192...","[4.19366169, 4.1924, 4.192973, 4.1950814, 4.18..."
1,Aix en Provence,43.529842,5.447474,0.843,"['Aix Homes ""Les Allées Provençales""', 'Aix Ho...",['https://www.booking.com/hotel/fr/aix-homes-1...,"[9.2, 8.7, 8.8, 9.1, 8.1, 8.7, 9.0, 9.1, 8.2, ...","['Set in Aix-en-Provence, the recently renovat...","[43.5256056, 43.5287484, 43.5272537, 43.537268...","[5.4409509, 5.4436653, 5.4452585, 5.4486186, 5..."
2,Amiens,49.894171,2.295695,0.679,['DOWNTOWN LOFT - CENTRE VILLE - WiFi - NETFLI...,['https://www.booking.com/hotel/fr/downtown-lo...,"[9.6, 9.0, 7.7, 10.0, 9.1, 8.2, 8.9, 9.0, 8.3,...","[""Situated 1.2 km from Amiens Train Station, 2...","[49.89219043, 49.8926758, 49.89125423, 49.8936...","[2.29471652, 2.3082578, 2.30433606, 2.301612, ..."
3,Annecy,45.899235,6.128885,0.683,['Highly recommended apartment steps from the ...,['https://www.booking.com/hotel/fr/annecy-vaca...,"[9.2, 9.4, 7.9, 8.6, 9.0, 8.2, 8.6, 9.0, 9.3, ...","[""Highly recommended apartment steps from the ...","[45.901008, 45.90584116, 45.88969452, 45.90393...","[6.126982, 6.14062912, 6.13690495, 6.11999363,..."
4,Avignon,43.949249,4.805901,0.870,"['My Pad Provence 6', 'Joli Studio Avec Jardin...",['https://www.booking.com/hotel/fr/my-pad-prov...,"[9.3, 8.5, 9.3, 9.3, 8.1, 8.0, 8.8, 8.6, 9.0, ...",['In the Avignon City Centre district of Avign...,"[43.9515236, 43.94182031, 43.9361955, 43.95065...","[4.8170918, 4.81495336, 4.82600075, 4.8033086,..."


In [202]:
# Insérez le DataFrame dans la base de données MySQL
try:
    df.to_sql(name='dbkayak', con=engine, if_exists='replace', index=False)
    print("DataFrame inséré avec succès dans la table 'db_kayak' de la base de données MySQL.")
except Exception as e:
    print(f"Erreur lors de l'insertion du DataFrame dans la base de données: {e}")

DataFrame inséré avec succès dans la table 'db_kayak' de la base de données MySQL.


In [203]:
from sqlalchemy import text

stmt = text("SELECT * FROM dbkayak.dbkayak LIMIT 5")

df = pd.read_sql_query(con=engine.connect(), sql=stmt)

df

,City,City_latitude,City_longitude,City_CCM,Hotels_name,Hotels_url,Hotels_score,Hotels_description,Hotels_latitude,Hotels_longitude
0,Aigues Mortes,43.566152,4.191540,0.896,"['Hôtel Le Médiéval', 'Maison Arthur', 'Au Cœu...",['https://www.booking.com/hotel/fr/le-medieval...,"[8.7, 9.0, 9.9, 9.5, 8.5, 9.2, 8.4, 9.1, 9.4, ...",['Le Médiéval is located on the banks of the c...,"[43.57186625, 43.5659428, 43.565401, 43.570192...","[4.19366169, 4.1924, 4.192973, 4.1950814, 4.18..."
1,Aix en Provence,43.529842,5.447474,0.843,"['Aix Homes ""Les Allées Provençales""', 'Aix Ho...",['https://www.booking.com/hotel/fr/aix-homes-1...,"[9.2, 8.7, 8.8, 9.1, 8.1, 8.7, 9.0, 9.1, 8.2, ...","['Set in Aix-en-Provence, the recently renovat...","[43.5256056, 43.5287484, 43.5272537, 43.537268...","[5.4409509, 5.4436653, 5.4452585, 5.4486186, 5..."
2,Amiens,49.894171,2.295695,0.679,['DOWNTOWN LOFT - CENTRE VILLE - WiFi - NETFLI...,['https://www.booking.com/hotel/fr/downtown-lo...,"[9.6, 9.0, 7.7, 10.0, 9.1, 8.2, 8.9, 9.0, 8.3,...","[""Situated 1.2 km from Amiens Train Station, 2...","[49.89219043, 49.8926758, 49.89125423, 49.8936...","[2.29471652, 2.3082578, 2.30433606, 2.301612, ..."
3,Annecy,45.899235,6.128885,0.683,['Highly recommended apartment steps from the ...,['https://www.booking.com/hotel/fr/annecy-vaca...,"[9.2, 9.4, 7.9, 8.6, 9.0, 8.2, 8.6, 9.0, 9.3, ...","[""Highly recommended apartment steps from the ...","[45.901008, 45.90584116, 45.88969452, 45.90393...","[6.126982, 6.14062912, 6.13690495, 6.11999363,..."
4,Avignon,43.949249,4.805901,0.870,"['My Pad Provence 6', 'Joli Studio Avec Jardin...",['https://www.booking.com/hotel/fr/my-pad-prov...,"[9.3, 8.5, 9.3, 9.3, 8.1, 8.0, 8.8, 8.6, 9.0, ...",['In the Avignon City Centre district of Avign...,"[43.9515236, 43.94182031, 43.9361955, 43.95065...","[4.8170918, 4.81495336, 4.82600075, 4.8033086,..."
